# ZenteiQ AI Tech Innovations — Task 3: MoE model (DeepSeek)

This notebook provides a detailed setup and execution workflow for training the **DeepSeek-16B-MOE** base model on a **CPU-only** environment using **MaxText** and **JAX** with synthetic data.

### 1. Installation & Environment Setup

In [1]:
# Clone the MaxText repository
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

# Install uv (fast package installer)
!pip install uv

# Set environmental variables to avoid GPU/TPU requirements
import os
os.environ["UV_TORCH_BACKEND"] = "cpu"

# Install MaxText CPU dependencies
!uv pip install -e .

# Restart the runtime after this step if run in Colab!

Cloning into 'maxtext'...
remote: Enumerating objects: 97124, done.
remote: Counting objects: 100% (2267/2267), done.
remote: Compressing objects: 100% (1153/1153), done.
remote: Total 97124 (delta 1850), reused 1128 (delta 1114), pack-reused 94857 (from 5)
Receiving objects: 100% (97124/97124), 411.75 MiB | 15.83 MiB/s, done.
Resolving deltas: 100% (70841/70841), done.
/content/maxtext
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 13.7 MB/s eta 0:00:0000:0100:01m
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 988ms                                          
Prepared 1 package in 2ms                                                
Installed 1 package in 3msm file:///content/maxtext)        
 + maxtext==0.2.3 (from file:///content/maxtext)


In [2]:
!uv pip install qwix tokamax -q

In [3]:
!uv pip install aqtp -q

### 2. Verify JAX CPU Backend Connection

Run the following cell to confirm that JAX successfully detects the CPU backend.

In [4]:
import jax
print("JAX Version:", jax.__version__)
print("Available devices:", jax.devices())
print("Default backend:", jax.default_backend())

JAX Version: 0.10.2
Available devices: [CpuDevice(id=0)]
Default backend: cpu


### 3. Configuration Breakdown

Below are the parameters we are setting for this training run, along with their purpose and effect:

| Configuration Parameter | Value | Purpose | Effect |
| :--- | :--- | :--- | :--- |
| **`model_name`** | `deepseek (scaled-down MoE)` | Specifies the base DeepSeek Mixture-of-Experts (MoE) architecture to use. | Loads the DeepSeek model configuration and initializes the transformer with MoE-specific components such as expert layers and routing mechanisms. |
| **`steps`** | `50` | Defines training duration. | Runs the optimization/training loop for exactly 50 iterations, which is sufficient to compile the graph, calculate device step times, and measure TFLOPs throughput. |
| **`dataset_type`** | `synthetic` | Determines the data loader pipeline format. | Bypasses external dataset downloading and disk I/O bottlenecks by dynamically generating random tensors of input tokens on CPU memory. This allows pure computational throughput benchmarking. |
| **`base_output_directory`** | `./output-deepseek` | Sets the file storage root path. | All metrics, TensorBoard execution logs, compile metadata, and model checkpoints are saved under this root directory. |
| **`run_name`** | `deepseek-cpu` | Uniquely names the specific execution run. | Prevents overwriting other runs and creates nested folders named `deepseek-cpu/` for output checkpoints and statistics. |

### 4. Execute CPU Training (Deepseel MOE 16B)

We run the main training script with our configuration overrides. We capture the stdout logs to parse metrics later.

In [5]:
!sudo apt-get update && sudo apt-get install -y \
    libxcomposite1 \
    libxcursor1 \
    libxdamage1 \
    libxext6 \
    libxfixes3 \
    libxi6 \
    libxrandr2 \
    libxrender1 \
    libxtst6 \
    libgbm1 \
    libasound2



Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]       
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Hit:13 https

In [4]:
# Run training for 50 steps on CPU
!python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml \
    model_name=deepseek2-16b\
    steps=50 \
    dataset_type=synthetic \
    base_output_directory=$(pwd)/cpu-output-deepseek2-16b \
    run_name=deepseek2-16b-cpu\
    megablox=false \
    override_model_config=true \
    skip_jax_distributed_system=true \
    base_emb_dim=1152 \
    base_mlp_dim=5120 \
    base_moe_mlp_dim=768 \
    base_num_decoder_layers=12 \
    num_experts=16 \
    num_experts_per_tok=2 \
    shared_experts=1 \
    per_device_batch_size=1 \
    attention=dot_product \
    max_target_length=512 \
    weight_dtype=bfloat16 \
    grad_dtype=bfloat16 \
    enable_checkpointing=false

2026-06-18 16:05:46.735007: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-18 16:05:48.303296: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-18 16:05:52.120252: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (ModuleNotFoundError)
[NO-OP] mldiagnostics: dependency missing; using stub. (

In [7]:
!uv pip install aqtp ml-goodput-measurement ml-collections -q

In [8]:
!uv pip install pathwaysutils aqt -q

In [9]:
!uv pip install drjax -q

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!cp -r cpu-output-deepseek2-16b /content/drive/MyDrive/
